# TECO: new three-label reconstruction at 40 Hz
CPU → Run all. This reads source EEG again and can take hours. Default: lexical = contextual-word = sentence label, content words only. Neutral retained; no class balancing. TeCo includes all task1 control **reading**. No claim about task2/3. Sparse neutral classes may prevent a group comparison. Old 30 Hz results remain separate. Saves compact means/onsets and small event-window features, not huge trial EEG. No GPU or GitHub token needed.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess, sys, os
from IPython.display import display, Markdown, Image
import pandas as pd
drive.mount('/content/drive')
THESIS=Path('/content/drive/MyDrive/Thesis')
REPO=Path('/content/word-sentiment-eeg-methods')
if not REPO.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/parmisbathayan/word-sentiment-eeg.git',str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
print('Code:', subprocess.check_output(['git','-C',str(REPO),'rev-parse','--short','HEAD'],text=True))
LABELS=THESIS/'gpt uploaded files/word_sentiment_eeg/teco/word_labels_three_level.csv'
# Only if the private revised file is missing, select the prepared CSV once.
import hashlib
EXPECTED_LABEL_SHA256='78f396181f0d79e2075c56a8e959fbb790b6251b0ef9cb7d53c36295a3b8f7a3'
if not LABELS.is_file():
    from google.colab import files
    print('Select word_labels_three_level.csv supplied privately by the project agent.')
    uploaded=files.upload()
    if len(uploaded)!=1:
        raise ValueError('Select exactly the revised TeCo label CSV.')
    payload=next(iter(uploaded.values()))
    if hashlib.sha256(payload).hexdigest()!=EXPECTED_LABEL_SHA256:
        raise ValueError('Selected CSV does not match the reviewed version; nothing saved to Drive.')
    LABELS.parent.mkdir(parents=True,exist_ok=True)
    LABELS.write_bytes(payload)
if hashlib.sha256(LABELS.read_bytes()).hexdigest()!=EXPECTED_LABEL_SHA256:
    raise ValueError('Drive label checksum differs; preserve it and ask the agent to reconcile versions.')
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(REPO/'requirements-colab.txt')],check=True)
RUN=THESIS/'Results/word_sentiment_eeg/teco_v2_three_level_40hz'
sys.path.insert(0,str(REPO/'scripts'))
from colab_progress import run_with_progress
command=[sys.executable,'-u',str(REPO/'run_three_level_erp.py'),'--dataset','teco','--thesis-root',str(THESIS),'--annotations',str(LABELS),'--save-event-features']
run_with_progress(command,RUN/'runtime_status.json',activity='New 40 Hz source reconstruction')


## Completion, class counts and comparison coverage

In [ ]:
import json
display(Markdown((RUN/'REPORT.md').read_text()))
display(pd.read_csv(RUN/'candidate_and_accepted_counts.csv'))
print(json.loads((RUN/'comparison_rosters.json').read_text()))


## task1: reader average, whole head and regions

In [ ]:
for p in sorted((RUN/'plots').glob('task1_all_three_group.png')):
    display(Image(filename=str(p)))
print('Whole-head signed voltage is near zero under average referencing; use the regional curves too.')


## task1: one reader per row; whole head followed by six regions

In [ ]:
from IPython.display import HTML
import base64
for p in sorted((RUN/'plots').glob('task1_*_all_three_row.png')):
    data=base64.b64encode(p.read_bytes()).decode()
    display(HTML(f'<div style="overflow-x:auto"><img style="min-width:1800px" src="data:image/png;base64,{data}"></div>'))
